--- 
# Partie 1 – Exploration du dataset

In [6]:
# Importer les packages
import os
import shutil
import hashlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile

from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

RAW_DIR = Path("../data/raw")
CLEANED_DIR = Path("../data/cleaned")
REPORTS_DIR = Path("../reports")

CLASSES = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
print("Classes détectées :", CLASSES)

Classes détectées : ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [7]:
def auditer_image(path: Path, classe: str) -> dict:
    """Extrait les métadonnées d'une image. Retourne un dict avec un flag 'corrompue' si le
    fichier ne peut pas être ouvert/lu correctement."""
    info = {
        "nom": path.name,
        "classe": classe,
        "chemin": str(path),
        "taille_octets": path.stat().st_size if path.exists() else np.nan,
        "format": np.nan,
        "mode": np.nan,
        "largeur": np.nan,
        "hauteur": np.nan,
        "nb_canaux": np.nan,
        "ecart_type_pixels": np.nan,
        "corrompue": False,
    }
    try:
        with Image.open(path) as im:
            im.verify() 
        with Image.open(path) as im:
            info["format"] = im.format
            info["mode"] = im.mode
            info["largeur"], info["hauteur"] = im.size
            arr = np.array(im.convert("RGB"))
            info["nb_canaux"] = len(im.getbands())
            info["ecart_type_pixels"] = float(arr.std())
    except Exception as e:
        info["corrompue"] = True
        info["erreur"] = str(e)
    return info


records = []
for classe in CLASSES:
    for path in sorted((RAW_DIR / classe).iterdir()):
        if path.is_file():
            records.append(auditer_image(path, classe))

audit_df = pd.DataFrame(records)
print(f"{len(audit_df)} images auditées.")
audit_df.head()


1032 images auditées.


,nom,classe,chemin,taille_octets,format,mode,largeur,hauteur,nb_canaux,ecart_type_pixels,corrompue,erreur
0,cardboard1.jpg,cardboard,..\data\raw\cardboard\cardboard1.jpg,17333,JPEG,RGB,512.0,384.0,3.0,40.586504,False,NaN
1,cardboard10.jpg,cardboard,..\data\raw\cardboard\cardboard10.jpg,21683,JPEG,RGB,512.0,384.0,3.0,42.577273,False,NaN
2,cardboard100.jpg,cardboard,..\data\raw\cardboard\cardboard100.jpg,14884,JPEG,RGB,512.0,384.0,3.0,46.121684,False,NaN
3,cardboard101.jpg,cardboard,..\data\raw\cardboard\cardboard101.jpg,14289,JPEG,RGB,512.0,384.0,3.0,72.264255,False,NaN
4,cardboard102.jpg,cardboard,..\data\raw\cardboard\cardboard102.jpg,18015,JPEG,RGB,512.0,384.0,3.0,48.389753,False,NaN


---
# Partie 2 – Détecter les images corrompues 

In [8]:
def est_corrompue(path: Path) -> bool:
    """Retourne True si l'image ne peut pas être ouverte/décodée correctement."""
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            im.load()
        return False
    except Exception:
        return True


images_corrompues = audit_df[audit_df["corrompue"]]
print(f"{len(images_corrompues)} images corrompues détectées :")
images_corrompues[["classe", "nom", "erreur"]]


6 images corrompues détectées :


,classe,nom,erreur
147,cardboard,cardboard83.jpg,cannot identify image file '..\\data\\raw\\car...
326,glass,glass74.jpg,Truncated File Read
446,metal,metal48.jpg,cannot identify image file '..\\data\\raw\\met...
633,paper,paper213.jpg,cannot identify image file '..\\data\\raw\\pap...
791,plastic,plastic13.jpg,cannot identify image file '..\\data\\raw\\pla...
1004,trash,trash3.jpg,cannot identify image file '..\\data\\raw\\tra...


---
# Partie 3 – Détecter les images vides

In [9]:
def est_vide(path: Path, seuil_std: float = 5.0) -> bool:
    """Retourne True si l'image est quasi uniforme (noire, blanche, ou très peu de variation)."""
    try:
        with Image.open(path) as im:
            arr = np.array(im.convert("RGB"))
        return arr.std() < seuil_std
    except Exception:
        return False 


images_valides = audit_df[~audit_df["corrompue"]].copy()
images_valides["vide"] = images_valides["chemin"].apply(lambda p: est_vide(Path(p)))

images_vides = images_valides[images_valides["vide"]]
print(f"{len(images_vides)} images quasi vides détectées :")
images_vides[["classe", "nom", "ecart_type_pixels"]]


4 images quasi vides détectées :


,classe,nom,ecart_type_pixels
167,cardboard,image-blanche-512x384.jpg,1.574664
355,glass,image-noire-512x384.png,0.000000
357,metal,image-blanche-512x384.jpg,1.574664
358,metal,image-noire-512x384.png,0.000000


---
# Partie 4 – Détecter les différences de résolution 

### 1) Déterminer la résolution minimale, la résolution maximale, les résolutions les plus fréquentes et le nombre d'images par résolution. 

In [10]:
images_ok = images_valides[~images_valides["vide"]].copy()
images_ok["resolution"] = list(zip(images_ok["largeur"].astype(int), images_ok["hauteur"].astype(int)))

res_min = (images_ok["largeur"].min(), images_ok["hauteur"].min())
res_max = (images_ok["largeur"].max(), images_ok["hauteur"].max())
print("Résolution minimale (largeur, hauteur) :", res_min)
print("Résolution maximale (largeur, hauteur) :", res_max)

print("\nNombre d'images par résolution (top 10) :")
res_counts = images_ok["resolution"].value_counts()
res_counts.head(10)


Résolution minimale (largeur, hauteur) : (32.0, 32.0)
Résolution maximale (largeur, hauteur) : (512.0, 384.0)

Nombre d'images par résolution (top 10) :


resolution
(512, 384)    1009
(32, 32)         5
(48, 32)         4
(40, 40)         4
Name: count, dtype: int64